# 🚀 Batch Runner — Execute All Project Notebooks
Runs every project notebook headlessly with **papermill**, one at a time, in this single Colab
session. A failure in one notebook is caught and logged — it does **not** stop the batch.

**Before running:**
1. `Runtime -> Change runtime type -> GPU`
2. Have your Kaggle API token (`kaggle.json`) ready — you'll upload it once, in Section 2, and
   every project notebook reuses it (no repeated upload prompts).
3. Point `CONFIG["repo_source"]` below at how you'll get the 29 project notebooks + this repo's
   folder structure into the Colab filesystem (Google Drive mount is the default — see Section 1).

**What this does NOT do:** it does not parallelize execution — Colab free tier gives you one GPU
runtime, so notebooks run strictly one after another. For true parallel execution you'd need
multiple Colab Pro+ runtimes or a local multi-GPU machine running this same loop with a process
pool instead of a plain `for` loop.


## 0. Setup

In [ ]:
!pip -q install papermill nbformat kaggle tqdm
import os, json, time, traceback
import pandas as pd
from datetime import datetime


## 1. Get the project folder into Colab
Pick ONE of the two options below and run only that cell.

In [ ]:
# --- Option A: mount Google Drive (recommended if you've uploaded PHD_Projects to Drive) ---
from google.colab import drive
drive.mount("/content/drive")

REPO_ROOT = "/content/drive/MyDrive/PHD_Projects"   # <-- EDIT to match where you placed the folder
assert os.path.isdir(REPO_ROOT), f"{REPO_ROOT} not found - upload/sync the PHD_Projects folder to this Drive path first."


In [ ]:
# --- Option B: clone from a git remote instead (skip this cell if you used Option A above) ---
# !git clone <your-repo-url> /content/PHD_Projects
# REPO_ROOT = "/content/PHD_Projects"


## 2. Kaggle authentication (once, reused by every notebook)

In [ ]:
from google.colab import files
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle auth ready.")


## 3. CONFIG — list of project notebooks to run

In [ ]:
PROJECT_PATHS = [
    "01_Electronic_Health_Record_Analysis/01_Electronic_Health_Record_Analysis.ipynb",
    "25_Multimodal_Disease_Digital_Twin/25_Multimodal_Disease_Digital_Twin.ipynb",
    "10_Suicidal_Ideation_Detection/10_Suicidal_Ideation_Detection.ipynb",
    "15_Drainage-Aware_Hyperlocal_Flood_Prediction/15_Drainage-Aware_Hyperlocal_Flood_Prediction.ipynb",
    "18_Medical_Symptom_Risk_Prediction/18_Medical_Symptom_Risk_Prediction.ipynb",
    "19_Explainable_Edge_Crop_Disease_Detection/19_Explainable_Edge_Crop_Disease_Detection.ipynb",
    "20_Stock_Price_Prediction/20_Stock_Price_Prediction.ipynb",
    "27_EHR_with_ML_DL_and_QML/27_EHR_with_ML_DL_and_QML.ipynb",
    "28_Complaint_and_Grievance_Management/28_Complaint_and_Grievance_Management.ipynb",
    "29_Unsupervised_Software_Anomaly_Detection/29_Unsupervised_Software_Anomaly_Detection.ipynb",
    "30_Traffic_Accident_Severity_Prediction/30_Traffic_Accident_Severity_Prediction.ipynb",
]
print(f"{len(PROJECT_PATHS)} project notebooks queued.")

In [ ]:
CONFIG = {
    "repo_root": REPO_ROOT,                                  # set in Section 1
    "output_dir": os.path.join(REPO_ROOT, "run_outputs"),    # executed notebooks land here
    "log_path": os.path.join(REPO_ROOT, "run_outputs", "batch_run_log.csv"),
    "timeout_sec": None,                                      # no cap - runtime varies a lot per notebook, unknown ahead of time
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Executed notebooks will be saved to: {CONFIG['output_dir']}")

## 4. Run all notebooks sequentially

In [ ]:
import papermill as pm
from google.colab import files

results = []
total = len(PROJECT_PATHS)

for i, rel_path in enumerate(PROJECT_PATHS, start=1):
    input_path = os.path.join(CONFIG["repo_root"], rel_path)
    output_path = os.path.join(CONFIG["output_dir"], os.path.basename(rel_path))
    project_no = os.path.basename(rel_path).split("_")[0]

    print(f"
[{i}/{total}] {project_no} - STARTING - {rel_path}")

    if not os.path.exists(input_path):
        print(f"[{i}/{total}] {project_no} - SKIPPED - file not found: {input_path}")
        results.append({"project": project_no, "path": rel_path, "status": "missing_file",
                         "seconds": 0, "error": "notebook not found at expected path"})
        continue

    start = time.time()
    try:
        pm.execute_notebook(
            input_path,
            output_path,
            kernel_name="python3",
            progress_bar=True,
            request_save_on_cell_execute=True,
            execution_timeout=CONFIG["timeout_sec"],
        )
        elapsed = time.time() - start
        print(f"[{i}/{total}] {project_no} - DONE in {elapsed:.0f}s -> {output_path}")
        results.append({"project": project_no, "path": rel_path, "status": "success",
                         "seconds": round(elapsed), "error": ""})

        # Download the executed (output-filled) notebook right away, so you get
        # proof-of-completion for each project as soon as it finishes.
        try:
            files.download(output_path)
        except Exception as dl_err:
            print(f"[{i}/{total}] {project_no} - WARNING - auto-download failed ({dl_err}); "
                  f"file is still saved at {output_path}")

    except Exception as e:
        elapsed = time.time() - start
        err_summary = str(e).splitlines()[-1][:300] if str(e) else type(e).__name__
        print(f"[{i}/{total}] {project_no} - FAILED after {elapsed:.0f}s - {err_summary}")
        results.append({"project": project_no, "path": rel_path, "status": "failed",
                         "seconds": round(elapsed), "error": err_summary})
        # Do NOT re-raise - continue to the next notebook so one failure doesn't kill the batch.

    done = i
    n_ok = sum(r["status"] == "success" for r in results)
    n_fail = sum(r["status"] == "failed" for r in results)
    print(f"[{i}/{total}] progress: {done} processed, {n_ok} succeeded, {n_fail} failed so far")

results_df = pd.DataFrame(results)
results_df.to_csv(CONFIG["log_path"], index=False)
results_df

In [ ]:
# Zip every executed notebook + the run log into one archive and download it in one shot -
# handy alternative to the per-notebook downloads above if you'd rather grab everything at once.
import shutil
zip_base = os.path.join(CONFIG["repo_root"], "run_outputs_bundle")
zip_path = shutil.make_archive(zip_base, "zip", CONFIG["output_dir"])
print(f"Bundled all executed notebooks + log into: {zip_path}")
files.download(zip_path)

## 5. Summary

In [ ]:
n_success = (results_df["status"] == "success").sum()
n_failed = (results_df["status"] == "failed").sum()
n_missing = (results_df["status"] == "missing_file").sum()

print(f"Succeeded: {n_success} / {len(results_df)}")
print(f"Failed:    {n_failed} / {len(results_df)}")
print(f"Missing:   {n_missing} / {len(results_df)}")

if n_failed > 0:
    print("\nFailed projects (see run_outputs/batch_run_log.csv and the individual executed")
    print("notebook in run_outputs/ for the full traceback of each):")
    print(results_df[results_df["status"] == "failed"][["project", "error"]].to_string(index=False))
